In [1]:
import torch
torch.__version__

'2.7.1+cpu'

In [2]:
import torch
torch.cuda.is_available()

False

### 1-Understanding tensor

In [3]:
tensor0d = torch.tensor(1)

tensor1d = torch.tensor([1, 2, 3])

tensor2d = torch.tensor([[1, 2, 3], 
                         [4, 5, 6]])

tensor3d = torch.tensor([[[1, 2], [3, 4]], 
                         [[5, 6], [8, 9]]])

In [4]:
print(tensor1d.dtype)

torch.int64


In [5]:
float_tensor = torch.tensor([1.0, 2.0, 3.0])
print(float_tensor.dtype)

torch.float32


32-bit floating-point number offers sufficient precision for most deep learning
tasks while consuming less memory and computational resources than a 64-bit floatingpoint
number. Moreover, GPU architectures are optimized for 32-bit computations, and
using this data type can significantly speed up model training and inference. Moreover, it is possible to change the precision using a tensor’s **'.to'** method.

In [6]:
float_tensor = float_tensor.to(torch.float32)
print(float_tensor.dtype)

torch.float32


In [7]:
print(tensor2d)

tensor([[1, 2, 3],
        [4, 5, 6]])


The **.shape** attribute allows us to access the shape of a tensor:

In [8]:
print(tensor2d.shape)

torch.Size([2, 3])


To reshape the tensor into a 3 × 2 tensor, we can use the **.reshape** method

In [9]:
print(tensor2d.reshape([3, 2]))

tensor([[1, 2],
        [3, 4],
        [5, 6]])


However, note that the more common command for reshaping tensors in PyTorch is
**.view()**

In [10]:
print(tensor2d.view([3, 2]))

tensor([[1, 2],
        [3, 4],
        [5, 6]])


The subtle difference between **.view()** and **.reshape()** in PyTorch lies in
their handling of memory layout: **.view()** requires the original data to be contiguous
and will fail if it isn’t, whereas **.reshape()** will work regardless, copying the data if necessary
to ensure the desired shape.

Oubliez les mathématiques un instant et imaginons comment fonctionne la **mémoire physique** de votre ordinateur (la RAM ou la VRAM).

#### a. Qu'est-ce que les "données physiques" ?
La mémoire de votre ordinateur n'a pas de concept de "tableau en 2D" ou de "cube en 3D". La mémoire est simplement une très longue **ligne droite de petites boîtes**.

Imaginons que vous créez un tenseur PyTorch simple, un tableau de 2 lignes et 2 colonnes :

```python
t = torch.tensor([[1, 2],
                  [3, 4]])
```

**Visuellement (Logique)**, vous voyez un carré. **Physiquement (dans la RAM)**, PyTorch range ces nombres dans 4 boîtes alignées l'une derrière l'autre : `[1, 2, 3, 4]`.

Quand PyTorch veut lire la première ligne `[1, 2]`, il lit les deux premières boîtes. Pour la deuxième ligne `[3, 4]`, il lit les deux suivantes. Puisque l'ordre de lecture logique correspond exactement à l'ordre des boîtes physiques, on dit que c'est **contigu** (bien rangé).

---

#### b. Comment un tenseur devient "non-contigu" (modifié de façon complexe) ?
Maintenant, appliquons une opération mathématique, par exemple la **transposition** (qui échange les lignes et les colonnes).

```python
t_transpose = t.t()
# Résultat logique (ce que vous voyez à l'écran) :
# [[1, 3],
#  [2, 4]]
```

Voici le secret de PyTorch pour être ultra-rapide : **il ne déplace absolument rien dans les boîtes physiques !** Dans la RAM, les boîtes sont toujours dans l'ordre : `[1, 2, 3, 4]`.

Pour vous afficher `[[1, 3], [2, 4]]`, PyTorch a juste ajouté une note interne qui dit : *"Attention, pour lire la première ligne, prends la boîte n°1, puis saute par-dessus la n°2 pour aller chercher la boîte n°3."*

Puisque PyTorch doit maintenant "sauter" d'une boîte à l'autre pour lire le tableau correctement, l'ordre de lecture ne correspond plus à l'ordre physique des boîtes. 
Le tenseur est devenu **non-contigu**. C'est cela une "modification complexe" : une opération qui change la forme logique sans toucher au rangement physique. (Couper un bout du tenseur avec du *slicing* comme `t[:, 1]` fait exactement la même chose).

---

#### c. Le drame avec `.view()`
La fonction `.view()` sert à donner une nouvelle forme à un tenseur (par exemple, transformer notre 2x2 en une ligne de 4). Mais `.view()` a une limitation technique : **elle refuse de travailler si elle doit sauter des boîtes.** Elle veut lire les boîtes sagement de gauche à droite.

Si vous faites `t_transpose.view(4)`, PyTorch panique : *"Hé, ce tenseur me demande de faire des sauts dans la mémoire physique, je ne peux pas faire un `.view()` là-dessus !"* -> **Erreur (Crash).**

---

#### d. Le côté magique de `.reshape()`
La fonction `.reshape(4)` est beaucoup plus intelligente. 
Quand vous la lancez sur `t_transpose` :

1. Elle regarde la mémoire physique et réalise : *"Ah, il y a des sauts, c'est non-contigu."*
2. Au lieu de crasher, elle va secrètement construire de **nouvelles boîtes physiques** quelque part ailleurs dans la RAM et les ranger dans le bon ordre direct : `[1, 3, 2, 4]`.
3. Maintenant que c'est propre, elle applique la nouvelle forme. -> **Succès, aucun crash.**

C'est pour ça qu'on dit que `.reshape()` "copie les données si nécessaire", et que c'est la méthode recommandée pour être tranquille !


We can use **.T** to transpose a tensor, which means flipping it across its diagonal

In [11]:
print(tensor2d.T)

tensor([[1, 4],
        [2, 5],
        [3, 6]])


The common way to multiply two matrices in PyTorch is the **.matmul** method

In [12]:
print(tensor2d.matmul(tensor2d.T))

tensor([[14, 32],
        [32, 77]])


### 2. Seeing models as computation graphs

PyTorch’s autograd system provides functions to compute gradients in dynamic computational
graphs automatically.
A computational graph is a directed graph that allows us to express and visualize
mathematical expressions. In the context of deep learning, a computation graph lays out the sequence of calculations needed to compute the output of a neural network—we will need this to compute the required gradients for backpropagation, the main
training algorithm for neural networks.

#### A logistic regression forward pass




<div align="center">
  <img src="../Introduction to PyTorch/A-7.png" width="600">
  <p><em>Logistic regression forward pass as a computation graph. The input feature x1 is multiplied by a model weight w1 and passed through an activation function σ after adding the bias. The loss is computed by comparing the model output a with a given label y.</em></p>
</div>




In [13]:
# This import statement is a common convention in Pytorch to prevent long lines of code
import torch.nn.functional as F

# True label
y = torch.tensor([1.0])

# Input feature
x1 = torch.tensor([1.1])

# Weight parameter
w1 = torch.tensor([2.2])

# Bias unit
b = torch.tensor([0.0])

# Net input
z1 = x1 * w1 + b

# Activation and output 
a = torch.sigmoid(z1)

loss = F.binary_cross_entropy(a, y)



### Automatic differentiation made easy

Si nous effectuons des calculs dans PyTorch, il construira par défaut un graphe de calcul en interne si l'un de ses nœuds terminaux possède l'attribut `requires_grad` défini sur `True`. Cela s'avère utile si nous souhaitons calculer des gradients. Les gradients sont indispensables lors de l'entraînement des réseaux de neurones via le célèbre algorithme de rétropropagation (ou *backpropagation*), que l'on peut considérer comme une application de la règle de dérivation en chaîne (*chain rule*) du calcul différentiel pour les réseaux de neurones, comme l'illustre la figure ci-dessous :

<div align="center">
  <img src="../Introduction to PyTorch/A-8.png" width="600">
  <p><em>The most common way of computing the loss gradients in a computation graph involves applying the chain rule from right to left, also called reverse-model automatic differentiation or backpropagation. We start from the output layer (or the loss itself) and work backward through the network to the input layer. We do this to compute the gradient of the loss with respect to each parameter (weights and biases) in the network, which informs how we update these parameters during training.</em></p>
</div>

- **Partial derivatives**, measure the rate at which a function changes with respect to one of its variables. 

- A **gradient** is a vector containing all of the
partial derivatives of a multivariate function (a function with more than one variable
as input)

`To put it simply, all you need to know is that the chain rule is a way to compute gradients of a loss function given the model’s parameters in a computation graph. This provides the information needed to update each parameter to minimize the loss function (which serves as a proxy for measuring the model’s performance using a method such as gradient descent).`

PyTorch’s **autograd engine** constructs a computational graph in the background by tracking every operation performed on tensors. Then, calling the **grad** function, we can compute the gradient of the
loss concerning the model parameter w1, as shown in the following listing.

In [14]:
import torch.nn.functional as F
from torch.autograd import grad

y = torch.tensor([1.0])
x1 = torch.tensor([1.1])
w1 = torch.tensor([2.2], requires_grad=True)
b = torch.tensor([0.0], requires_grad=True)

z1 = x1 * w1 + b
a = torch.sigmoid(z1)

loss = F.binary_cross_entropy(a, y)


By default, Pytorch destroys the computations graph after calculating the gradients to free memory. However, since we will reuse this computation graph shortly, we set **retain_graph=True** so that it stays in memory.

In [15]:

grad_L_w1 = grad(loss, w1, retain_graph=True)
grad_L_b = grad(loss, b, retain_graph=True)

In [16]:
print(grad_L_w1)
print(grad_L_b)

(tensor([-0.0898]),)
(tensor([-0.0817]),)


Here, we have been using the grad function manually, which can be useful for experimentation, debugging, and demonstrating concepts. 


But, in practice, PyTorch provides
even more high-level tools to automate this process. For instance, we can call **.backward** on the loss, and PyTorch will compute the gradients of all the leaf nodes in the graph, which will be stored via the tensors’ **.grad** attributes

In [17]:
loss.backward()
print(w1.grad)
print(b.grad)

tensor([-0.0898])
tensor([-0.0817])


### Implementing a multilayer neural networks

Let’s look at a multilayer perceptron, a fully connected neural network, as illustrated in the figure below :


<div align="center">
  <img src="../Introduction to PyTorch/A-9.png" width="600">
  <p><em>A multilayer perceptron with two hidden layers. Each node representsa unit in the respective layer. For illustration purposes, each layer has a very small number of nodes.</em></p>
</div>

When implementing a neural network in PyTorch, we can subclass the `torch.nn.Module` class to define our own custom network architecture. This Module base class provides a lot of functionality, making it easier to build and train models. For instance, it allows us to encapsulate layers and operations and keep track of the model’s parameters.

Within this subclass, we define the network layers in the `__init__` constructor and specify how the layers interact in the forward method. 

The **forward method** describes how the input data passes through the network and comes together as a computation graph. 
In contrast, the **backward method**, which we typically do not need to implement ourselves, is used during training to compute gradients of the loss function given the model parameters

In [18]:
class NeuralNetwork(torch.nn.Module):
    """Coding the number of inputs and oututs as variables allows us to reuse the same code for datasets with different numbers of features and classes."""

    def __init__(self, num_inputs, num_outputs):
        super().__init__()

        self.layers = torch.nn.Sequential(
            # 1st hidden layer 
            torch.nn.Linear(num_inputs, 30), # The Linear layer takes the number of input and output nodes as arguments
            torch.nn.ReLU(), # Nonlinear activation functions are placed between the hidden layers

            # 2nd hidden layer
            torch.nn.Linear(30, 20), # The number of output nodes of one hidden layer has to match the number of inputs of the next layer
            torch.nn.ReLU(),

            # output layer
            torch.nn.Linear(20, num_outputs), 
        )

    def forward(self, x):
        logits = self.layers(x)
        return logits # The outputs of the last layers are called logits

`torch.nn.Sequential` n'est pas obligatoire. C'est simplement un raccourci très pratique ("une boîte") pour regrouper des couches qui s'exécutent les unes à la suite des autres en ligne droite.

L'alternative classique, qui est même la méthode la plus courante consiste à définir chaque couche individuellement, puis à relier les tuyaux vous-même dans la fonction `forward()`.

In [19]:
class NeuralNetwork(torch.nn.Module):

    def __init__(self, num_inputs, num_outputs):
        super().__init__()

        # On définit toutes nos couches comme des attributs séparés
        self.layer1 = torch.nn.Linear(num_inputs, 30)
        self.relu1 = torch.nn.ReLU()
        
        self.layer2 = torch.nn.Linear(30, 20)
        self.relu2 = torch.nn.ReLU()
        
        self.output_layer = torch.nn.Linear(20, num_outputs)

    def forward(self, x):
        # On fait passer manuellement les données 'x' d'une couche à l'autre
        x = self.layer1(x)
        x = self.relu1(x)
        
        x = self.layer2(x)
        x = self.relu2(x)
        
        logits = self.output_layer(x)
        
        return logits

In [20]:
# We can now create an instance of our NeuralNetwork class and print it to see its architecture.
model = NeuralNetwork(50, 30)
print(model)

NeuralNetwork(
  (layer1): Linear(in_features=50, out_features=30, bias=True)
  (relu1): ReLU()
  (layer2): Linear(in_features=30, out_features=20, bias=True)
  (relu2): ReLU()
  (output_layer): Linear(in_features=20, out_features=30, bias=True)
)


##### 1. `model.parameters()`
Méthode obtenu en héritant de `torch.nn.Module`, quand nous appelons `model.parameters()`, PyTorch parcourt de lui-même **toutes les couches** de notre réseau (nos `nn.Linear`, qu'elles soient dans un `Sequential` ou non). 

Pour chaque couche `Linear`, il "récupère" deux choses essentielles :
1.  **La matrice des Poids (*weights*)** (qui relie l'entrée à la sortie).
2.  **Le vecteur des Biais (*biases*)**
*(Les fonctions d'activation comme `ReLU` n'ont aucun poids ni paramètre, elles sont justes ignorées !)*

Il nous renvoie une liste (un générateur, plus exactement) contenant chacun de ces gros tableaux mathématiques (les tenseurs).

##### 2. `for p in ...`
La boucle parcourt cette liste. À chaque tour, `p` est un tenseur (une matrice de poids ou un vecteur de biais).

##### 3. `p.numel()`
Comme nous l'avons vu, il compte le nombre total de valeurs à l'intérieur du tenseur `p`.
Par exemple :
*   Si `p` est la matrice de poids de la 1re couche (`Linear(50, 30)`), il compte $50 \times 30 = 1500$ éléments.
*   Si `p` est le biais de cette 1re couche, il compte $30$ éléments.
*   Total pour cette couche : 1530 paramètres.

##### 4. `if p.requires_grad` (Le détail crucial)
C'est un filtre très astucieux.
Il dit à la boucle : "Ne compte que les paramètres que l'optimiseur a le droit de modifier pendant l'entraînement (`requires_grad=True`)".



In [21]:
# To calculate the total number of trainable parameters in the model, we can use the following code:
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Total number of trainable model parameters:", num_params)

Total number of trainable model parameters: 2780


In [22]:
# We can access the corresponding weight parameter matrix as dollows :
print(model.layer1.weight)

Parameter containing:
tensor([[-7.1704e-05, -1.0923e-01, -8.8778e-02,  ...,  9.7147e-02,
          3.0375e-02,  1.1321e-01],
        [-1.3421e-01,  2.6556e-02, -1.2063e-01,  ...,  2.0907e-02,
          7.1527e-02, -5.4063e-02],
        [-9.4758e-02,  8.5760e-02, -8.7431e-02,  ...,  1.0654e-01,
         -1.4026e-01,  1.3639e-01],
        ...,
        [-4.4982e-02, -1.0809e-01,  1.3871e-02,  ...,  3.7008e-02,
          1.2033e-01,  6.4161e-02],
        [-4.5979e-02, -6.5983e-02, -4.7220e-02,  ...,  1.2423e-01,
         -4.3480e-02,  5.7004e-02],
        [-3.7906e-02, -5.3428e-02, -9.8200e-02,  ..., -9.8972e-02,
          1.3627e-02,  3.6078e-02]], requires_grad=True)


In [23]:
# lets use the .shape to show it dimension
print(model.layer1.weight.shape)

torch.Size([30, 50])


In [24]:
# We can access the bais vector via :
print(model.layer1.bias)

Parameter containing:
tensor([ 0.1249,  0.0399, -0.0680,  0.0138, -0.0296, -0.0681,  0.1104, -0.0476,
         0.0127, -0.1116,  0.0623, -0.0810,  0.0297, -0.0841, -0.0806, -0.1311,
        -0.0143, -0.0601,  0.1071, -0.1116, -0.0550,  0.0758, -0.0100,  0.1245,
        -0.0592,  0.1311,  0.0668, -0.0977, -0.0717, -0.0635],
       requires_grad=True)


#### We can make the random number initialization reproducible by seeding Pytorch's random generator via `manual_seed`

In [25]:
torch.manual_seed(123)
model = NeuralNetwork(50, 3)
print(model.layer1.weight)

Parameter containing:
tensor([[-0.0577,  0.0047, -0.0702,  ...,  0.0222,  0.1260,  0.0865],
        [ 0.0502,  0.0307,  0.0333,  ...,  0.0951,  0.1134, -0.0297],
        [ 0.1077, -0.1108,  0.0122,  ...,  0.0108, -0.1049, -0.1063],
        ...,
        [-0.0787,  0.1259,  0.0803,  ...,  0.1218,  0.1303, -0.1351],
        [ 0.1359,  0.0175, -0.0673,  ...,  0.0674,  0.0676,  0.1058],
        [ 0.0790,  0.1343, -0.0293,  ...,  0.0344, -0.0971, -0.0509]],
       requires_grad=True)


#### Let's briefly see how NeuralNetwork is used via the forward pass 

In [26]:
torch.manual_seed(123)
X = torch.rand(1, 50)
out = model(X)
print(out)

tensor([[-0.1262,  0.1080, -0.1792]], grad_fn=<AddmmBackward0>)


##### The forward pass refers to calculating output tensors from input tensors. This involves passing the input data through all the neural network layers, starting from the input layer, through hidden layers, and finally to the output layer.

##### These three numbers returned here correspond to a score assigned to each of the three output nodes. Notice that the output tensor also includes a `grad_fn value`

##### Here, `grad_fn =<AddmmBackward0>` represents the last-used function to compute a variable in the computational graph. PyTorch will use this information when it computes gradients during backpropagation. In this case, it is an Addmm operation. **Addmm** stands for **matrix multiplication (mm)** followed by an **addition (Add)**.

##### When we use a model **for inference (for instance, making predictions) rather than training**, the best practice is to use the `torch.no_grad()` context manager. This tells PyTorch that it doesn’t need to keep track of the gradients, which can result in significant savings in memory and computation.

In [27]:
with torch.no_grad():
    out = model(X)
print(out)

tensor([[-0.1262,  0.1080, -0.1792]])


In PyTorch, it’s **common practice** to code models such that they return the outputs of the last layer (logits) **without passing them to a nonlinear activation function**. That’s because PyTorch’s commonly used loss functions combine the softmax (or sigmoid for binary classification) operation with the negative log-likelihood loss in a single class. The reason for this is **numerical efficiency and stability**.
So, if we want to **compute class-membership probabilities** for our predictions, we have to call the softmax function **explicitly**.

In [28]:
with torch.no_grad():
    out = torch.softmax(model(X), dim=1)
print(out)

tensor([[0.3113, 0.3934, 0.2952]])


### Setting up efficient data loaders 

The overall idea behind data loading in PyTorch is illustrated in the figure below :

<div align="center">
  <img src="../Introduction to PyTorch/A-10.png" width="600">
  <p><em>PyTorch implements a Dataset and a DataLoader class. The Dataset class is used to instantiate objects that define how each data record is loaded. The DataLoader handles how the data is shuffled and assembled into batches.</em></p>
</div>

In [29]:
X_train = torch.tensor([
    [-1.2, 3.1],
    [-0.9, 2.9],
    [-0.5, 2.6],
    [2.3, -1.1],
    [2.7, -1.5]
])

y_train = torch.tensor([0, 0, 0, 1, 1])

X_test = torch.tensor([
    [-0.8, 2.8],
    [2.6, -1.6]
])

y_test = torch.tensor([0, 1])

`NB :` PyTorch requires that class labels start with label 0, and the largest
class label value should not exceed the number of output nodes minus 1
(since Python index counting starts at zero). So, if we have class labels 0, 1, 2,
3, and 4, the neural network output layer should consist of five nodes.

In PyTorch, the three main components of a custom Dataset class are the
`__init__` constructor, the `__getitem__` method, and the `__len__` method.


- In the `__init__` method, we set up attributes that we can access later in the
`__getitem__` and `__len__` methods. These could be file paths, file objects, database connectors, and so on.

- In the `__getitem__` method, we define instructions for returning exactly one item
from the dataset via an index. This refers to the features and the class label corresponding to a single training example or test instance.

- Finally, the `__len__` method contains instructions for retrieving the length of the dataset. Here, we use the `.shape` attribute of a tensor to return the number of rows in the feature array.

In [30]:
from torch.utils.data import Dataset, DataLoader

class ToyDataset(Dataset):
    def __init__(self, X, y):
        self.features = X
        self.labels = y
    
    # Instructions for retrieving exactly one data record and the corresponding label
    def __getitem__(self, index):
        one_x = self.features[index]
        one_y = self.labels[index]
        return one_x, one_y
    
    # Instructions for returning the length of the dataset
    def __len__(self):
        return self.labels.shape[0]
    

train_ds = ToyDataset(X_train, y_train)
test_ds = ToyDataset(X_test, y_test)

In [31]:
print(len(train_ds))

5


In [32]:
from torch.utils.data import DataLoader

torch.manual_seed(123)

# The ToyDataset insatance created earlier serves aas input in the data loader
train_loader = DataLoader(
    dataset=train_ds,
    batch_size=2,
    shuffle=True, # Wether or not to shuffle the data
    num_workers=0 # The number of background processes
)

test_loader = DataLoader(
    dataset=test_ds,
    batch_size=2,
    shuffle=False, # It is not necessary to shuffle a test dataset
    num_workers=0
)

`shuffle=True` sert à mélanger l’ordre des exemples à chaque époque d’entraînement. Son intérêt est surtout d’**empêcher le modèle d’apprendre un biais lié à l’ordre des données**. En pratique, ça aide l’optimisation avec SGD/minibatch, rend l’apprentissage plus stable, et améliore souvent la généralisation.

On l’utilise surtout pendant le training parce que le modèle doit voir les exemples dans un ordre différent à chaque passage. Si les données sont ordonnées par classe, difficulté, temps, ou source, ne pas mélanger peut rendre l’apprentissage moins bon.

On ne l’utilise généralement pas pendant le test ou la validation, parce qu’on veut une évaluation déterministe et reproductible. L’ordre des exemples ne doit pas influencer la mesure de performance. Donc on garde `shuffle=False` pour tester, valider, et comparer les résultats de façon stable.

Cas où il faut faire attention: si tes données sont séquentielles ou temporelles, comme une série temporelle ou certains problèmes de langage, il peut être incorrect de mélanger certains éléments. Dans ce cas, on choisit un autre type de découpage ou de batching.

En résumé:
- training: souvent `shuffle=True`
- validation/test: presque toujours `shuffle=False`
- exception: données séquentielles ou dépendantes de l’ordre

In [33]:
for idx, (x, y) in enumerate(train_loader):
    print(f"Batch {idx + 1}: ", x, y)

Batch 1:  tensor([[ 2.3000, -1.1000],
        [-0.9000,  2.9000]]) tensor([1, 0])
Batch 2:  tensor([[-1.2000,  3.1000],
        [-0.5000,  2.6000]]) tensor([0, 0])
Batch 3:  tensor([[ 2.7000, -1.5000]]) tensor([1])


Comme on peut le voir à partir de la sortie précédente, `train_loader` parcourt le dataset d’entraînement en visitant chaque exemple une seule fois. C’est ce qu’on appelle une époque d’entraînement.

Comme nous avons initialisé le générateur aléatoire avec `torch.manual_seed(123)` ici, vous devriez obtenir exactement le même ordre de mélange des exemples d’entraînement. En revanche, si vous parcourez le dataset une deuxième fois, vous verrez que l’ordre du mélange change.

C’est voulu, afin d’éviter que les réseaux de neurones profonds ne se retrouvent bloqués dans des cycles de mise à jour répétitifs pendant l’entraînement.

We specified a batch size of 2 here, but the third batch only contains a single example.
That’s because we have five training examples, and 5 is not evenly divisible by 2.

In practice, **having a substantially smaller batch as the last batch in a training epoch can disturb the convergence during training**. To prevent this, set `drop_last=True`, **which will drop the last batch in each epoch**, as shown in the following listing.

In [34]:
train_loader = DataLoader(
    dataset=train_ds,
    batch_size=2,
    shuffle=True,
    num_workers=0,
    drop_last=True
)

In [35]:
for idx, (x, y) in enumerate(train_loader):
    print(f"Batch {idx}:", x, y)

Batch 0: tensor([[-1.2000,  3.1000],
        [-0.5000,  2.6000]]) tensor([0, 0])
Batch 1: tensor([[ 2.3000, -1.1000],
        [-0.9000,  2.9000]]) tensor([1, 0])


-------------------------------------

L'apprentissage profond (*deep learning*) repose sur un mariage subtil entre des abstractions logicielles de haut niveau et une compréhension fine du matériel informatique sous-jacent. Lorsqu'un ingénieur écrit `DataLoader(dataset, num_workers=4, batch_size=32)`, il mobilise en réalité un écosystème complet mettant en jeu des processus systèmes, des mécanismes de communication inter-processus, des hiérarchies mémoire, et des unités de calcul aux architectures radicalement différentes.

En tant que débutant, il est bien normal de se demander : **comment les données transitent-elles, du stockage disque jusqu'aux cœurs de calcul d'un GPU, et comment ce pipeline peut-il être optimisé lorsque l'on dispose de plusieurs unités de traitement ?**

Pour répondre à cette question, nous procéderons en quatre temps. Nous étudierons d'abord l'architecture du `DataLoader` de PyTorch et le rôle des *workers*. Nous analyserons ensuite la répartition naturelle des tâches entre CPU et GPU, en fondant cette répartition sur les propriétés architecturales de ces deux types de processeurs. Nous examinerons le paradigme du parallélisme CPU et la manière dont PyTorch l'exploite. Enfin, nous nous pencherons sur les stratégies d'entraînement multi-GPU, qui constituent le sommet de la complexité dans ce domaine.

---

#### Partie I — Le DataLoader et les Workers : anatomie d'un pipeline de données

##### 1.1 La boucle d'entraînement et son goulot d'étranglement fondamental

Un réseau de neurones s'entraîne par itérations successives. À chaque itération, le modèle reçoit un *batch* de données, calcule une prédiction (*forward pass*), mesure son erreur via une fonction de perte (*loss*), puis rétropropage cette erreur à travers ses couches pour ajuster ses paramètres (*backward pass*). Cette séquence se répète des milliers, voire des millions de fois.

Or, cette boucle possède une structure temporelle asymétrique : le GPU, lorsqu'il effectue un *forward* et un *backward pass*, travaille à une vitesse considérable — de l'ordre de quelques millisecondes pour un batch sur du matériel moderne. En revanche, **charger ce batch depuis le disque, le décoder, lui appliquer des transformations, et l'assembler en tenseur** peut prendre un temps comparable, voire supérieur.

Si ces deux phases sont exécutées séquentiellement — c'est-à-dire si le GPU attend que les données soient prêtes avant de commencer à calculer — alors le GPU est en réalité **inactif la moitié du temps**. C'est ce qu'on appelle un *I/O bottleneck* (goulot d'étranglement sur les entrées/sorties). Pour un équipement qui coûte plusieurs milliers d'euros, ce gaspillage est inacceptable.

La solution conceptuelle est celle du **pipeline** : préparer les prochains batchs *pendant* que le GPU travaille sur le batch courant. C'est exactement ce que réalise le mécanisme des *workers*.

##### 1.2 Architecture interne du DataLoader

Le `DataLoader` de PyTorch n'est pas un simple itérateur. C'est un **orchestrateur** qui coordonne plusieurs composants distincts, chacun ayant une responsabilité précise.

**Le Dataset** est la couche d'abstraction qui représente les données. Il expose une interface minimale : `__len__()` pour connaître le nombre d'éléments, et `__getitem__(i)` pour accéder à l'élément d'indice `i`. C'est dans cette méthode que réside toute la logique de chargement : ouverture de fichier, décodage d'image, lecture depuis une base de données, etc. Le `Dataset` est volontairement ignorant de la notion de batch — il travaille sur des échantillons individuels.

**Le Sampler** est responsable de définir *l'ordre* dans lequel les indices seront parcourus. Le `SequentialSampler` les parcourt dans l'ordre naturel (0, 1, 2, ..., N-1). Le `RandomSampler`, activé par `shuffle=True`, génère une permutation aléatoire de ces indices à chaque nouvelle époque. Cette randomisation est cruciale pour l'entraînement : elle empêche le modèle de mémoriser l'ordre des données et améliore la généralisation.

**Le BatchSampler** est une couche supérieure au-dessus du Sampler. Il regroupe les indices générés par le Sampler en sous-listes de taille `batch_size`. Si `drop_last=True`, le dernier groupe est ignoré lorsqu'il contient moins de `batch_size` éléments.

**La collate_fn** est la fonction qui, étant donné une liste d'échantillons individuels, les assemble en un batch tensoriel. Par défaut, PyTorch fournit une `collate_fn` qui empile les tenseurs individuels le long d'un nouvel axe de dimension zéro. Il est possible de fournir une fonction personnalisée pour gérer des structures de données complexes (dictionnaires, objets à taille variable, etc.).

**Les workers** sont les processus fils chargés d'exécuter `dataset[i]` et d'appliquer les transformations, en parallèle du processus principal.

##### 1.3 Le mode `num_workers=0` : tout dans le processus principal

Lorsque `num_workers=0` (valeur par défaut), il n'y a aucun processus fils. Le processus principal est seul, et il effectue **toutes les opérations séquentiellement** :

```
[Main Process]
  1. Demander les indices au BatchSampler → [3, 1]
  2. Appeler dataset[3] → charger et décoder l'échantillon 3
  3. Appeler dataset[1] → charger et décoder l'échantillon 1
  4. Appeler collate_fn([échantillon_3, échantillon_1]) → batch
  5. Transférer le batch vers le GPU
  6. Exécuter le forward pass
  7. Exécuter le backward pass
  8. Mettre à jour les poids
  9. Retourner en 1.
```

Dans ce mode, le GPU est **inactif** aux étapes 1 à 5. Il attend que le CPU ait fini de préparer les données.

Ce mode est cependant parfaitement adapté aux datasets de petite taille entièrement chargés en RAM (comme dans notre exemple jouet avec 5 tenseurs), car le coût de chargement est négligeable, et les workers introduiraient un surcoût de démarrage inutile.

##### 1.4 Le mode `num_workers > 0` : parallélisme par processus

Dès que `num_workers=N` avec N > 0, PyTorch crée N processus fils au démarrage de l'itération. Ces processus sont créés via le mécanisme de **fork** (ou *spawn* sur Windows et macOS selon la configuration), qui est un appel système dupliquant l'espace mémoire du processus parent.

Le flux de contrôle devient alors le suivant :

```
[Main Process]
  ├── Crée N workers au démarrage
  ├── Génère tous les groupes d'indices via le BatchSampler
  ├── Dépose ces groupes dans une Index Queue (file partagée)
  ├── Attend dans la Result Queue qu'un batch soit prêt
  └── Quand un batch arrive → GPU forward/backward → mise à jour

[Worker 1] (processus fils indépendant)
  ├── Attend dans l'Index Queue
  ├── Prend un groupe d'indices [i, j]
  ├── Appelle dataset[i] et dataset[j]
  ├── Applique les transforms
  ├── Appelle collate_fn
  └── Dépose le batch dans la Result Queue

[Worker 2] (idem, en parallèle)
  └── ...
```

Les files `Index Queue` et `Result Queue` sont des files de type `multiprocessing.Queue`, qui utilisent des *pipes* système pour transmettre des données entre processus. Les tenseurs y sont sérialisés via **shared memory** (mémoire partagée), ce qui évite de copier physiquement les données : seule une référence mémoire est transmise.

Le paramètre `prefetch_factor` (par défaut 2) contrôle combien de batchs chaque worker prépare à l'avance. Avec `num_workers=4` et `prefetch_factor=2`, jusqu'à 8 batchs peuvent être en cours de préparation simultanément — bien avant que le GPU en ait besoin.

<div align="center">
  <img src="../Introduction to PyTorch/A-11.png" width="800">
  <p><em>Loading data without multiple workers (setting num_workers=0) will create a data loading bottleneck where the model sits idle until the next batch is loaded (left). If multiple workers are enabled, the data loader can queue up the next batch in the background (right).</em></p>
</div>

##### 1.5 Le mécanisme de *pin_memory* 

Pour bien cerner l'utilité du paramètre `pin_memory=True`, il convient d'abord de lever une confusion stricte entre les deux mémoires fondamentales d'un système. D'une part, le **disque dur (ou SSD)** fait office d'armoire d'archivage : sa capacité est immense et pérenne, mais son accès est lent. D'autre part, la **mémoire vive (RAM)** constitue l'espace de travail immédiat du processeur : sa capacité est très restreinte, mais sa vitesse de lecture et d'écriture est extrêmement élevée.

Lorsque les tâches s'accumulent (comme l'ouverture de multiples applications ou le traitement de vastes tenseurs), cet espace de travail qu'est la RAM arrive rapidement à saturation. Pour éviter la paralysie de l'ordinateur, le système d'exploitation utilise un mécanisme de prévention (la *mémoire virtuelle* ou le *swapping*). Il déplace de manière autonome, secrète et continuelle les données inactives de la RAM vers un espace de relève situé sur le disque dur. L'espace de travail principal est ainsi désengorgé.

Toutefois, cette fluidité de la gestion de la mémoire pose un véritable défi lors de l'entraînement d'un réseau de neurones avec une carte graphique (GPU). Pendant l'entraînement, le CPU prépare les lots d'images (batchs) et les dispose dans la RAM. Cependant, le GPU étant un périphérique ultra-rapide mais aveugle aux réarrangements du système d'exploitation, exige que les données soient parfaitement fixes. Si le système d'exploitation décidait soudainement de déplacer le lot d'images vers le disque dur pour libérer de la place, le transfert vers le GPU échouerait ou serait excessivement ralenti.

C'est là le rôle du paramètre **`pin_memory=True`** (la mémoire épinglée ou *page-locked memory*). L'activer équivaut à clouer les lots de données dans une zone strictement intouchable de la RAM, interdisant formellement au système d'exploitation de les déplacer vers le disque dur.

**L'intérêt pour l'optimisation est double :** 
Une fois les données verrouillées physiquement en RAM, la carte graphique est autorisée à venir les y piocher en totale autonomie grâce à un raccourci matériel nommé **DMA** (*Direct Memory Access*). Le transfert s'opère donc à très haut débit et devient totalement asynchrone : pendant que le GPU lit les données, le processeur central (CPU) est dispensé de participer au transfert et peut d'ores et déjà préparer le lot suivant.

**Que se passe-t-il si l'on n'active pas cette option ?**
Le travail est doublé. Puisque les données résident dans un espace instable, le CPU est forcé de s'arrêter pour copier d'abord les images de l'espace standard vers un compartiment verrouillé temporaire qu'il détruit ensuite. Cette étape intermédiaire monopolise les ressources du CPU et génère un goulot d'étranglement ralentissant tout l'entraînement.

---

#### Partie II — CPU et GPU : deux architectures pour deux missions

##### 2.1 Le CPU : un généraliste puissant et séquentiel

Un processeur central moderne (CPU) est conçu autour d'un paradigme de **latence minimale** : son objectif est d'exécuter une instruction donnée aussi vite que possible. Pour y parvenir, il embarque des mécanismes sophistiqués :

- **Pipeline superscalaire** : exécution simultanée de plusieurs instructions d'une même séquence à différents stades
- **Exécution dans le désordre** (*out-of-order execution*) : réorganisation des instructions pour éviter les attentes
- **Prédiction de branchement** : anticipation du résultat d'un `if/else` pour ne pas interrompre le pipeline
- **Caches hiérarchiques** (L1, L2, L3) : réduction de la latence d'accès mémoire par stockage local des données fréquemment utilisées
- **Quelques cœurs très puissants** : typiquement 4 à 64 cœurs sur du matériel grand public à haut de gamme

Cette architecture fait du CPU un outil idéal pour les **tâches séquentielles à logique complexe** : lire un fichier depuis le disque (appel système impliquant le noyau OS), décoder un JPEG (algorithme DCT avec de nombreuses branches conditionnelles), appliquer des augmentations de données (redimensionnement, rotation, découpage aléatoire...).

##### 2.2 Le GPU : un spécialiste du calcul massivement parallèle

Un GPU est conçu autour d'un paradigme radicalement différent : le **débit maximal** (*throughput*). Son objectif n'est pas d'exécuter une instruction rapidement, mais d'exécuter des **millions d'instructions similaires simultanément**.

Un GPU NVIDIA RTX 4090 possède par exemple **16 384 cœurs CUDA**. Ces cœurs sont regroupés en blocs (*Streaming Multiprocessors*), et sont organisés pour exécuter le modèle SIMT (*Single Instruction, Multiple Threads*) : un seul flux d'instructions est exécuté par des milliers de threads simultanément, chacun opérant sur une donnée différente.

Cette architecture est parfaitement adaptée aux **opérations matricielles** : dans une multiplication de deux matrices A (m×k) et B (k×n), chaque élément du résultat C (m×n) est calculé indépendamment des autres. Un GPU peut donc calculer tous ces éléments simultanément, là où un CPU les calculerait séquentiellement.

Or, l'apprentissage profond est essentiellement du calcul matriciel : une couche *fully connected* est une multiplication matricielle, une convolution est une multiplication matricielle restructurée, un mécanisme d'attention (*transformer*) est une suite de multiplications matricielles. Le GPU est donc l'outil naturel pour les *forward* et *backward passes*.

##### 2.3 Pourquoi le chargement de données ne peut pas être délégué au GPU

On pourrait naïvement envisager de confier le chargement des données au GPU. Cette idée se heurte à plusieurs obstacles fondamentaux.

**Premièrement, l'accès aux fichiers système.** Lire un fichier sur disque implique un *syscall* (appel au noyau du système d'exploitation). Le GPU n'a pas d'accès direct au système de fichiers — il est isolé dans son propre espace d'adressage (VRAM), sans capacité d'effectuer des opérations I/O autonomes. Toute lecture disque passe nécessairement par le CPU et la RAM.

**Deuxièmement, la capacité mémoire limitée.** La VRAM d'un GPU est limitée : 8 Go à 80 Go selon les modèles. Un dataset d'images courant (ImageNet : 150 Go, LAION : plusieurs To) ne peut pas y tenir intégralement. Le CPU joue donc le rôle de *feeder* : il lit les données depuis le disque vers la RAM, puis les transfère vers la VRAM au fur et à mesure.

**Troisièmement, la nature des opérations de prétraitement.** Le décodage JPEG est un algorithme à base de transformation discrète en cosinus (DCT) avec de nombreuses étapes conditionnelles et une structure de données arborescente (Huffman). Les augmentations de données (rotations, *crops* aléatoires, *color jitter*...) impliquent de la logique conditionnelle et des accès mémoire non uniformes. Ces caractéristiques sont **défavorables** à l'architecture SIMT du GPU, qui pénalise fortement la divergence de branchement entre threads.

Le CPU est donc non pas une limitation imposée, mais bien l'outil **architecturalement adapté** à ces tâches.

---

#### Partie III — Parallélisme CPU : théorie et mécanismes

##### 3.1 Les deux formes de parallélisme CPU dans PyTorch

Il est crucial de distinguer deux niveaux de parallélisme CPU, qui opèrent à des granularités différentes.

**Le parallélisme inter-processus** est celui que réalisent les workers. Il s'agit de faire exécuter plusieurs tâches indépendantes par plusieurs processus OS distincts, chacun s'exécutant sur un cœur CPU différent. C'est du **vrai parallélisme** — au sens où plusieurs instructions machines sont exécutées littéralement au même instant sur des unités physiques distinctes.

Python, de par son *Global Interpreter Lock* (GIL), interdit le vrai parallélisme au sein d'un même processus : deux threads Python ne peuvent jamais exécuter du bytecode Python simultanément. La solution est donc de contourner le GIL en utilisant des **processus séparés** plutôt que des threads. C'est pourquoi PyTorch utilise `multiprocessing` et non `threading` pour ses workers.

**Le parallélisme intra-opération** est celui que réalisent les bibliothèques de calcul numérique. Quand PyTorch exécute une opération matricielle sur CPU (`torch.mm`, `torch.conv2d`...), il fait appel en interne à des bibliothèques optimisées comme **Intel MKL** (Math Kernel Library) ou **OpenBLAS**, qui décomposent l'opération en sous-tâches et les répartissent sur l'ensemble des cœurs CPU via **OpenMP** ou des threads natifs. Ce parallélisme est transparent pour le développeur.

##### 3.2 La mécanique du fork et la duplication du Dataset

Lorsque PyTorch crée un worker avec `num_workers=N`, il fait appel à `torch.multiprocessing.Process`, qui repose sur `multiprocessing.Process` de la bibliothèque standard. Sur Linux, le mécanisme utilisé par défaut est `fork`.

Le `fork` est un appel système qui crée une copie exacte du processus parent. L'espace mémoire entier est dupliqué — y compris le Dataset et toutes ses ressources. Cependant, grâce au mécanisme **Copy-on-Write** (COW) du noyau Linux, cette duplication est paresseuse : les pages mémoire ne sont effectivement copiées que lorsqu'un des processus les modifie. Si le Dataset est en lecture seule (ce qui est le cas en général), les workers partagent physiquement la même mémoire que le parent, sans surcoût.

Sur macOS et Windows, le mécanisme par défaut est `spawn` (ou `forkserver`), qui crée un processus vierge et y réimporte les modules nécessaires. C'est plus lent au démarrage mais plus sûr (pas de problèmes liés aux ressources héritées comme les connexions réseau ou les descripteurs de fichiers).

##### 3.3 La communication inter-processus et la sérialisation

Les workers et le processus principal communiquent via des **queues multiprocessing**, qui sont implémentées au-dessus de *pipes* Unix. L'envoi d'un objet dans une queue implique sa **sérialisation** (via `pickle`) d'un côté, et sa **désérialisation** de l'autre.

Pour les petits objets (indices, métadonnées), ce mécanisme est efficace. Pour les tenseurs volumineux, PyTorch utilise une optimisation majeure : la **mémoire partagée**. Plutôt que de sérialiser le tenseur dans le pipe, le worker l'alloue dans un segment de mémoire partagée (`/dev/shm` sur Linux) et ne transmet au processus principal qu'un **handle** (une référence) vers ce segment. Le processus principal accède alors directement aux données sans copie physique.

##### 3.4 L'équilibre entre le nombre de workers et les ressources système

Augmenter `num_workers` n'est pas toujours bénéfique. Plusieurs facteurs limitants entrent en jeu.

Le nombre de **cœurs CPU disponibles** constitue la limite naturelle du parallélisme. Avoir plus de workers que de cœurs entraîne de la contention : les workers se disputent les cœurs, et le *context switching* (bascule entre processus) introduit une surcharge.

La **bande passante disque** est souvent le vrai goulot d'étranglement. Si les données sont sur un disque dur (HDD), la tête de lecture ne peut servir qu'une requête à la fois. Multiplier les workers qui lisent simultanément provoque des mouvements de tête désordonnés (*seek*), réduisant la bande passante effective. Sur SSD NVMe, le parallélisme I/O est bien mieux supporté.

La **consommation mémoire** augmente linéairement avec le nombre de workers : chaque worker maintient en mémoire `prefetch_factor` batchs, plus une copie du Dataset.

La règle empirique est d'utiliser entre 2 et 8 workers pour un entraînement sur images, en testant les performances réelles avec un profileur.

---

#### Partie IV — Entraînement Multi-GPU : stratégies et mécanismes

---

##### 4.1 Pourquoi plusieurs GPUs ?

Pour comprendre l'utilité de plusieurs GPUs, il faut d'abord saisir deux limites fondamentales et bien distinctes que l'on rencontre lors de l'entraînement de modèles modernes.

La première limite est une limite **physique de stockage**. Chaque GPU dispose d'une mémoire vidéo (VRAM) dont la capacité est fixe et non extensible — typiquement entre 16 Go et 80 Go selon les modèles haut de gamme. Or, un grand modèle de langage comme GPT-3 pèse à lui seul 350 Go en précision float16. Il est donc tout simplement *impossible*, au sens physique du terme, de loger l'intégralité de ses paramètres dans un seul GPU. Peu importe la rapidité du matériel, le problème n'est pas de vitesse mais de capacité d'accueil : on ne peut pas faire tenir un meuble de cinq mètres dans une pièce de deux mètres. Il faut alors **distribuer le modèle** lui-même sur plusieurs GPUs, chacun n'en hébergeant qu'une fraction.

La seconde limite est une limite **de temps**. Même lorsque le modèle tient confortablement dans un seul GPU, l'entraînement peut durer des semaines ou des mois. Ici, l'enjeu n'est plus la capacité mais la vitesse. Si l'on dispose de N GPUs capables de traiter des données en parallèle, on peut théoriquement diviser le temps d'entraînement par N en traitant N fois plus d'exemples à chaque instant.

Ces deux motivations — l'une contraignante, l'autre optimisante — donnent naissance à deux grandes familles de stratégies que nous allons explorer : le **parallélisme de données**, où le modèle est dupliqué et les données sont réparties ; et le **parallélisme de modèle**, où c'est le modèle lui-même qui est fractionné entre les GPUs.

---

##### 4.2 DataParallel (DP) : la première approche, naïve

`torch.nn.DataParallel` est l'API historique de PyTorch pour l'entraînement multi-GPU. Pour bien comprendre ses limites, il faut d'abord comprendre comment elle fonctionne de l'intérieur — et cela commence par deux notions fondamentales : ce qu'est un **processus** et ce qu'est un **thread**.

---

**Processus et threads : poser les bases.**

Lorsque vous lancez un programme Python, le système d'exploitation crée un **processus**. Ce processus est une unité d'exécution totalement autonome et isolée : il possède son propre espace mémoire, ses propres ressources, sa propre identité. Deux processus ne partagent rien par défaut. Si l'un plante, l'autre continue de tourner sans en être affecté.

Un **thread** est quelque chose de plus fin. C'est un fil d'exécution secondaire qui vit *à l'intérieur* d'un même processus. Plusieurs threads coexistent dans le même processus, partagent le même espace mémoire et les mêmes ressources, mais avancent chacun dans le code à leur propre rythme.

L'analogie la plus parlante est celle d'un **bureau et de ses employés**. Le processus est le bureau : il a ses propres murs, ses propres dossiers, son propre matériel. Les threads sont les employés qui travaillent dans ce bureau. Ils ont tous accès aux mêmes armoires, aux mêmes fichiers, aux mêmes outils — c'est précisément ce partage qui leur permet de collaborer rapidement, sans avoir à s'échanger des copies de documents. Mais ce même partage crée un risque : si deux employés modifient le même fichier en même temps, le résultat sera corrompu.

C'est exactement pour éviter cette corruption que Python a introduit le **GIL** (*Global Interpreter Lock*). Imaginez qu'il n'existe qu'**un seul stylo** dans le bureau, et qu'un employé doit l'avoir en main pour avoir le droit d'agir. Les autres peuvent attendre, observer, se préparer — mais ils ne peuvent rien *exécuter* tant que le stylo n'est pas libre.

Conséquence directe et non intuitive : même si votre machine possède 8 cœurs physiques capables d'exécuter 8 fils d'exécution simultanément, Python ne laissera jamais deux threads s'exécuter *vraiment* en même temps. L'un tient le stylo, les autres attendent leur tour. Le parallélisme apparent est donc une **illusion** : on a l'impression d'une exécution simultanée, mais les threads se relaient en réalité en alternance très rapide.

---

**Le fonctionnement de DataParallel.**

`DataParallel` s'appuie précisément sur ce modèle de threads. Pour comprendre pourquoi c'est problématique, suivons le déroulement d'une itération d'entraînement.

Le **processus principal** — un bureau unique avec un seul chef — prend en charge l'ensemble du batch depuis le DataLoader. Il le découpe en portions égales et en confie une à chacun des GPUs disponibles. Chaque GPU effectue alors, en parallèle, un *forward pass* indépendant sur sa portion. Jusqu'ici, l'organisation semble efficace.

Mais voici où le premier problème se révèle : une fois les calculs terminés sur chaque GPU, **toutes les sorties sont renvoyées au GPU 0** pour qu'il les rassemble, calcule la perte globale, effectue le *backward pass*, puis redistribue les gradients et resynchronise les poids de tous les GPUs. Autrement dit, la phase de travail parallèle est éphémère — le reste de l'itération est entièrement séquentiel et concentré sur une seule machine.

Ce modèle crée un **goulot d'étranglement structurel** sur le GPU 0. Il est le seul à recevoir le batch complet, à assembler les résultats, à calculer les gradients globaux, et à redistribuer les poids. Il est donc, en permanence, bien plus sollicité que ses homologues. En pratique, au-delà de 2 à 4 GPUs, les GPUs supplémentaires n'apportent presque aucun gain réel : ils passent la majorité de leur temps à attendre que le GPU 0 ait terminé.

---

**Le second défaut : le GIL paralyse la coordination.**

Un second problème, moins visible mais tout aussi pénalisant, découle directement de l'architecture à threads décrite plus haut. Dans `DataParallel`, les différents GPUs sont pilotés par des threads Python distincts, tous logés dans le **même processus**. L'intention est louable : faire avancer plusieurs GPUs en parallèle. Mais dès que ces threads ont besoin d'exécuter du code Python pur — interpréter une boucle, évaluer une condition, appeler une fonction personnalisée dans le *forward pass* — ils se retrouvent à faire la queue devant le GIL.

Pendant que le thread du GPU 0 tient le stylo, le thread du GPU 1 est suspendu, quand bien même son GPU est matériellement disponible et impatient de travailler. Plus le *forward pass* du modèle contient de logique Python complexe, plus cette file d'attente s'allonge — et plus le bénéfice du multi-GPU s'évapore. La coordination entre threads, au lieu d'accélérer les choses, finit par les freiner.

Ces deux défauts — la centralisation sur le GPU 0 et le GIL — ne sont pas des bugs corrigeables par un patch : ils sont la conséquence directe du choix architectural fondateur de `DataParallel`, celui d'un processus unique pilotant tout. C'est ce choix qu'il fallait remettre en cause, et c'est exactement ce que fait DDP.

---

##### 4.3 DistributedDataParallel (DDP) : le standard actuel

<div align="center">
  <img src="../Introduction to PyTorch/A-12.png" width="800">
  <p><em>The model and data transfer in DDP involves two key steps. First, we create a copy of the model on each of the GPUs. 
         Then we divide the input data into unique minibatches that we pass on to each model copy.</em></p>
</div>

---

<div align="center">
  <img src="../Introduction to PyTorch/A-13.png" width="800">
  <p><em>The forward and backward passes in DDP are executed independently on each GPU with its corresponding data subset. 
  Once the forward and backward passes are completed, gradients from each model replica (on each GPU) are synchronized across all GPUs. 
  This ensures that every model replica has the same updated weights.</em></p>
</div>

---

Pour comprendre pourquoi `DistributedDataParallel` (DDP) représente une rupture architecturale et non une simple amélioration, il faut revenir sur la source du problème précédent : la centralisation. `DataParallel` avait un chef unique. DDP supprime entièrement cette hiérarchie.

**De la cuisine centralisée à la brigade décentralisée.**

Avec DDP, chaque GPU est géré par son **propre processus Python indépendant**, lancé via `torchrun`. Il n'existe plus de processus maître vers lequel tout converge. Chaque processus charge ses propres données, calcule ses propres gradients, et prend ses propres décisions — tout en restant parfaitement synchronisé avec ses pairs.
Cette indépendance des processus résout immédiatement le problème du GIL : puisqu'il s'agit désormais de processus distincts.

**Chaque processus charge ses propres données.**

Pour éviter que tous les GPUs ne traitent les mêmes exemples — ce qui serait un gaspillage total — PyTorch fournit un `DistributedSampler`. Son rôle est de partitionner le dataset en sous-ensembles disjoints, un par processus. Si l'on dispose de N processus et M exemples au total, le processus numéro k se voit attribuer les exemples d'indices k, k+N, k+2N, etc. L'ensemble des processus couvre ainsi exactement l'intégralité du dataset, sans redondance ni omission.

**La synchronisation des gradients par AllReduce.**

À l'issue du *backward pass*, chaque processus a calculé ses propres gradients — exacts pour son mini-batch, mais partiels au regard de l'ensemble des données. Pour que tous les modèles évoluent de façon identique et convergent vers la même solution, il est impératif que chaque GPU applique **la même mise à jour**, c'est-à-dire la moyenne des gradients calculés sur l'ensemble des processus.

C'est précisément la fonction de l'opération **AllReduce**. Elle garantit que, quel que soit le GPU interrogé, chacun reçoit la somme (ou la moyenne) complète de tous les gradients calculés par ses pairs. Cette opération est implémentée par **NCCL** (*NVIDIA Collective Communications Library*), qui exploite les connexions physiques directes entre GPUs : NVLink pour les GPUs sur la même machine, InfiniBand ou Ethernet pour les GPUs répartis sur des serveurs distants.

**L'algorithme Ring-AllReduce : pourquoi un anneau ?**

L'implémentation naïve d'un AllReduce consisterait à envoyer tous les gradients vers un nœud central qui les somme puis les redistribue. On retomberait alors dans le goulot d'étranglement de DataParallel. NCCL évite ce piège grâce au **Ring-AllReduce**, dont l'élégance mérite une explication.

Voici une explication reformulée, construite directement depuis le contenu du papier.

---

**Le Ring-AllReduce : comment N GPUs s'échangent leurs gradients sans goulot d'étranglement**

Dans l'entraînement distribué en données parallèles, chaque GPU calcule ses propres gradients sur son sous-ensemble du batch. Il faut ensuite les moyenner entre tous les GPUs avant de mettre à jour les poids. Mais le défaut est immédiat : le GPU central doit recevoir les gradients de tous les autres, puis les renvoyer à tous. Sa charge de communication croît linéairement avec le nombre de GPUs. Avec un modèle de 300 millions de paramètres (soit 1,2 Go de gradients) et 10 GPUs, chaque itération se ralentit de plus de 10 secondes. La solution ne passe pas à l'échelle.

Le **Ring-AllReduce** élimine ce goulot en supprimant le réducteur central. Voici comment il fonctionne.

**La topologie.** On dispose les N GPUs en anneau logique. Chaque GPU a exactement un voisin à sa gauche et un à sa droite. Il n'envoie des données qu'à droite, il n'en reçoit que depuis la gauche.

**La première phase : le Scatter-Reduce.** Chaque GPU découpe son tableau de gradients en N fragments de taille égale. Puis on fait N-1 tours dans l'anneau. À chaque tour, chaque GPU envoie un fragment à son voisin de droite et reçoit un fragment de son voisin de gauche — qu'il **additionne** à son propre fragment correspondant. Le fragment envoyé à chaque tour est toujours celui reçu au tour précédent. Au bout de N-1 tours, chaque GPU détient un fragment qui contient la **somme complète** de ce fragment, agrégée sur l'ensemble des GPUs. Pas l'intégralité des gradients — juste sa portion à lui, mais parfaitement réduite.

**La deuxième phase : l'Allgather.** On refait N-1 tours dans l'anneau, mais cette fois chaque GPU, au lieu d'additionner ce qu'il reçoit, **écrase** simplement le fragment correspondant avec la valeur reçue. Au bout de N-1 tours, chaque GPU a reçu successivement tous les fragments réduits et possède donc l'intégralité des gradients agrégés.

**Pourquoi la bande passante est indépendante de N.** À chaque tour, chaque GPU envoie et reçoit un fragment de taille K/N (où K est la taille totale du tableau). Sur les 2(N-1) tours au total (N-1 par phase), la quantité totale de données transférée par chaque GPU est donc 2(N-1) × K/N, ce qui tend vers 2K quand N est grand — et surtout **ne dépend pas de N**. Peu importe qu'on ait 4 ou 400 GPUs dans l'anneau, chaque lien transporte la même quantité de données. C'est la propriété fondamentale qui rend l'algorithme scalable : ajouter des GPUs n'aggrave pas la communication.

**L'optimisation supplémentaire.** Puisque la rétropropagation calcule les gradients depuis la dernière couche vers la première, les gradients des couches de sortie sont disponibles bien avant ceux des couches d'entrée. On peut donc démarrer le Ring-AllReduce sur les premiers gradients disponibles pendant que les autres sont encore en cours de calcul, chevauchant communication et calcul. Dans les expériences du papier sur un modèle de 300 millions de paramètres, cela permettait d'économiser 70 à 120 ms par itération.

<div>
    Pour plus de détails, vous pouvez consulter 
    <a href="../Introduction to PyTorch/baidu_allreduce_oct6.pdf">
        Bringing HPC Techniques to Deep Learning
    </a>.
</div>

---

##### 4.4 Parallélisme de modèle (*Model Parallelism*)

Aussi efficace que soit DDP, il suppose implicitement une chose : que le modèle tient dans la VRAM d'un seul GPU. Lorsque ce n'est plus le cas — ce qui est la norme pour les très grands modèles — une toute autre famille de stratégies s'impose, dans laquelle ce n'est plus le dataset qui est distribué, mais **le modèle lui-même**.

**Le pipeline parallelism : la chaîne de montage.**

La forme la plus intuitive de parallélisme de modèle consiste à assigner des groupes de couches à des GPUs différents. Le GPU 0 héberge les premières couches, le GPU 1 les suivantes, et ainsi de suite. Lors d'un *forward pass*, les activations sont calculées sur le GPU 0, transmises au GPU 1 pour la suite du calcul, puis au GPU 2, etc. L'analogie avec une chaîne de montage industrielle est directe : chaque station effectue une opération précise et passe le résultat à la suivante.

Mais cette analogie révèle aussi immédiatement le défaut majeur de l'approche naïve. Sur une vraie chaîne de montage où une seule pièce circule à la fois, toutes les stations sont inactives sauf une. C'est exactement ce qui se passe ici : à chaque instant, un seul GPU travaille pendant que tous les autres attendent. L'utilisation effective est de 1/N — catastrophique.

La solution consiste à ne plus faire circuler un seul batch, mais à le découper en **micro-batchs** injectés en rafale dans le pipeline. Dès que le GPU 0 a traité le micro-batch 1 et l'a transmis au GPU 1, il n'attend pas le résultat final : il attaque immédiatement le micro-batch 2. Pendant ce temps, le GPU 1 reçoit le micro-batch 1 et commence à le traiter. Le GPU 2 fait de même un cycle plus tard. Les premiers cycles servent à remplir le pipeline — c'est inévitable, comme le début de toute chaîne de montage. Mais une fois ce remplissage terminé, tous les GPUs travaillent en permanence, chacun sur un micro-batch différent au même instant. Cette technique, introduite par GPipe et PipeDream, a rendu l'entraînement des très grands modèles économiquement viable.
<div align="center">
  <img src="../Introduction to PyTorch/Pipeline parallelism diagram.png" width="1000">
</div>


**Le tensor parallelism : découper les matrices elles-mêmes.**

Le pipeline parallelism distribue les couches *entre* les GPUs, mais que faire lorsqu'une seule couche — une matrice de projection dans un mécanisme d'attention, par exemple — est à elle seule trop volumineuse pour un GPU ?

Le **tensor parallelism** répond à cette question en distribuant les **matrices de poids elles-mêmes** entre GPUs. Concrètement, une grande matrice peut être découpée soit par colonnes, soit par lignes. Chaque GPU ne stocke et ne calcule qu'un fragment de cette matrice, produisant un résultat partiel. Ces résultats partiels sont ensuite agrégés via AllReduce pour reconstituer la sortie complète. Cette approche, popularisée par **Megatron-LM** (voir l'article <a href="../Introduction to PyTorch/Megatron-LM-Training Multi-Billion Parameter Language Models Using.pdf">Megatron-LM-Training Multi-Billion Parameter Language Models Using</a>) de NVIDIA, opère donc *au sein* d'une même couche, là où le pipeline parallelism opérait *entre* couches.

**Le 3D Parallelism : la synthèse des trois stratégies.**

En pratique, les modèles d'envergure comme GPT-4 ou LLaMA 3 ne choisissent pas entre ces stratégies : ils les **combinent toutes trois simultanément**, dans une organisation appelée *3D Parallelism* (voir l'article <a href="../Introduction to PyTorch/Efficient Large-Scale Language Model Training on GPU Clusters Using Megatron-LM.pdf">Efficient Large-Scale Language Model Training on GPU Clusters Using Megatron-LM</a>).

- Le **parallélisme de données** (DDP) opère entre groupes de GPUs : plusieurs répliques du même sous-modèle traitent des mini-batchs différents et synchronisent leurs gradients.
- Le **parallélisme de pipeline** opère entre étages de couches : les différents blocs du modèle sont répartis sur des GPUs distincts mis en cascade.
- Le **parallélisme de tenseurs** opère *au sein* de chaque couche : les matrices sont fragmentées entre GPUs d'un même étage.

Chaque dimension de ce cube résout un problème différent : le tensor parallelism gère les couches trop larges, le pipeline parallelism gère les modèles trop profonds, et le data parallelism maximise le débit. Leur combinaison est ce qui rend techniquement possible l'entraînement de modèles comptant des centaines de milliards de paramètres sur des clusters de milliers de GPUs.

---

*Autres ressources très intéressantes :*

- <a href="https://lilianweng.github.io/posts/2021-09-25-train-large/#data-parallelism">How to Train Really Large Models on Many GPUs?</a>

- <a href="https://huggingface.co/spaces/nanotron/ultrascale-playbook?section=high-level_overview">The Ultra-Scale Playbook: Training LLMs on GPU Clusters</a>

### A typical training loop